# 03 — Séries temporelles Sentinel-2

Construction de la table spatio-temporelle SeineCrops à partir des scènes
Sentinel-2 L2A cataloguées en sprint S1 (`02_disponibilite_s2.ipynb`).

Ce notebook couvre quatre étapes séquentielles du sprint S2 :

1. Téléchargement de la bande SCL (60 m) et calcul de la disponibilité
   effective sur l'AOI (`f_valid_aoi`) pour chaque scène du catalogue dédupliqué.
2. Téléchargement des bandes spectrales retenues (B02, B04, B05, B06, B07,
   B08, B11) pour les scènes dont `f_valid_aoi ≥ 0.01` ; resampling bilinéaire de toutes les bandes à 20 m.
3. Calcul des indices (NDVI, EVI, NDWI, NDRE) et composite mensuel par médiane.
4. Agrégation zonale sur les parcelles RPG (mean, std, p10, p90) et chargement
   dans `derived.s2_parcelles_monthly` (PostGIS).

---

## AOI et tuiles

| Paire | Tuiles | Zone couverte |
|-------|--------|---------------|
| Nord  | 30UYA, 31UCR | Pays de Caux (rive droite) |
| Sud   | 30UYV, 31UCQ | Plateau de Neubourg (rive gauche) |

---

## Bandes et indices

| Bande / Indice | Type | Résolution native | Formule |
|----------------|------|-------------------|---------|
| B02 | Bande | 10 m → 20 m | — |
| B04 | Bande | 10 m → 20 m | — |
| B05 | Bande | 20 m | — |
| B06 | Bande | 20 m | — |
| B07 | Bande | 20 m | — |
| B08 | Bande | 10 m → 20 m | — |
| B11 | Bande | 20 m | — |
| NDVI | Indice | — | (B08 − B04) / (B08 + B04) |
| EVI  | Indice | — | 2.5 × (B08 − B04) / (B08 + 6×B04 − 7.5×B02 + 1) |
| NDWI | Indice | — | (B08 − B11) / (B08 + B11) |
| NDRE | Indice | — | (B08 − B05) / (B08 + B05) |

Toutes les bandes sont resamplées à 20 m (bilinéaire) avant calcul
des indices. Le composite est calculé par médiane mensuelle sur les scènes
dont `f_valid_aoi ≥ 0.01`.

---

## Feature set résultant

11 variables (7 bandes + 4 indices) × 4 statistiques (mean, std, p10, p90)
× 16 mois = **704 features** par parcelle, stockées dans
`derived.s2_parcelles_monthly`.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| 3.1 | Téléchargement SCL et calcul `f_valid_aoi` — complétion du catalogue dédupliqué |
| 3.2 | Téléchargement bandes spectrales et calcul des indices — scènes retenues (`f_valid_aoi ≥ 0.01`) |
| 3.3 | Composite mensuel — médiane par mois × bande/indice × tuile |
| 3.4 | Agrégation zonale et chargement PostGIS — `derived.s2_parcelles_monthly` |

---

## Références

- [Documentation CDSE — téléchargement produits S2](https://documentation.dataspace.copernicus.eu/APIs/OData.html)
- [Sentinel-2 L2A — bande SCL (classes et usage)](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html)
- [HR-VPP — manuel produit trajectoires saisonnières (PDF)](https://land.copernicus.eu/en/technical-library/product-user-manual-of-seasonal-trajectories/@@download/file)
- [BreizhCrops — feature set de référence pour la classification de cultures](https://github.com/dl4sits/BreizhCrops)
- `02_disponibilite_s2.ipynb` — sprint S1, catalogue CDSE, `df_dedup`, `AVAILABILITY_REPORT.json`
- `01_ingestion_rpg.ipynb` — sprint S1, `derived.rpg_parcelles_aoi` (géométries de référence)

### 3.1 — Téléchargement SCL et calcul `f_valid_aoi`

Téléchargement de la bande SCL (60 m) pour chaque scène du catalogue
dédupliqué et calcul de la fraction de pixels valides sur l'AOI
(`f_valid_aoi`).

`f_valid_aoi` est la métrique de disponibilité effective : contrairement au
`cloud_cover_catalogue` (couverture nuageuse déclarée sur la tuile entière),
elle mesure précisément la fraction de pixels exploitables **sur l'AOI**,
en excluant les classes SCL invalides :

| Classe SCL | Signification |
|------------|---------------|
| 3 | Ombres nuageuses |
| 8 | Nuages moyennement probables |
| 9 | Nuages hautement probables |
| 10 | Cirrus |
| 11 | Neige / glace |

Seules les scènes dont `f_valid_aoi ≥ 0.01` (au moins 1 % de pixels valides
sur l'AOI) sont retenues pour le téléchargement des bandes spectrales en S3.2.

Les fichiers SCL sont stockés dans `data/raw/s2/scl/<tile_id>/` et non
versionnés (`.gitignore`).

**Dépendance** : `data/raw/s2/catalogue_dedup.parquet` produit par
`02_disponibilite_s2.ipynb` — ré-exécuter nb02 si le fichier est absent.

In [ ]:
# ── Imports (communs à toutes les sections) ───────────────────────────
import os
import gc
import time
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import pyproj
import rasterio
import rasterio.mask
import rasterio.warp
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.features import rasterize as rio_rasterize
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from pyproj import CRS
import psycopg2
from psycopg2.extras import execute_values

# ── PROJ : utiliser la base de l'environnement Python ─────────────────
# (et non celle de PostgreSQL, incompatible avec pyproj)
os.environ["PROJ_DATA"]    = pyproj.datadir.get_data_dir()
os.environ["PROJ_LIB"]     = pyproj.datadir.get_data_dir()
os.environ["PROJ_NETWORK"] = "OFF"

from pyproj import CRS as ProjCRS

# ── Racine du projet ──────────────────────────────────────────────────
def find_project_root(marker: str = ".projectroot") -> Path:
    here = Path().resolve()
    for parent in [here, *here.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Racine du projet introuvable")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

# ── Constantes ────────────────────────────────────────────────────────
ODATA_BASE_CAT  = "https://catalogue.dataspace.copernicus.eu/odata/v1"   # catalogue
ODATA_BASE_DL   = "https://download.dataspace.copernicus.eu/odata/v1"    # téléchargement / Nodes
F_VALID_SEUIL  = 0.01                                                    # fraction minimale de pixels valides sur l'AOI
SCL_INVALIDES  = {1, 3, 7, 8, 9, 10, 11}                                 # classes SCL exclues du calcul f_valid_aoi
SCL_DIR        = PROJECT_ROOT / "data" / "raw" / "s2" / "scl"
PARQUET_PATH   = PROJECT_ROOT / "data" / "raw" / "s2" / "catalogue_dedup.parquet"
AOI_PATH       = PROJECT_ROOT / "data" / "vector" / "aoi" / "aoi_seinecrops.geojson"
MAX_WORKERS    = 1             # parallélisme téléchargement SCL

SCL_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SCL_DIR      : {SCL_DIR}")
print(f"PROJ_DATA    : {os.environ['PROJ_DATA']}")

In [ ]:
# ── Chargement du catalogue dédupliqué ───────────────────────────────
if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {PARQUET_PATH}\n"
        "Ré-exécuter 02_disponibilite_s2.ipynb pour le générer."
    )

df_dedup = pd.read_parquet(PARQUET_PATH)
print(f"Scènes chargées : {len(df_dedup)}")
print(f"Colonnes        : {list(df_dedup.columns)}")
print(f"Tuiles          : {df_dedup['tile_id'].unique().tolist()}")

# Normalisation : scene_id sans suffixe .SAFE (le champ Name OData
# peut ou non inclure .SAFE selon la version du catalogue)
df_dedup["scene_id"] = df_dedup["scene_id"].str.removesuffix(".SAFE")

# ── Chargement de l'AOI ───────────────────────────────────────────────
aoi = gpd.read_file(AOI_PATH)
print(f"\nAOI CRS         : {aoi.crs}")
print(f"AOI géométries  : {len(aoi)}")

In [ ]:
# ── Authentification CDSE (OAuth2) ───────────────────────────────────
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

def get_cdse_token() -> dict:
    resp = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "password",
            "client_id":  "cdse-public",
            "username":   os.environ["CDSE_USER"],
            "password":   os.environ["CDSE_PASSWORD"],
        },
        timeout=30,
    )
    resp.raise_for_status()
    token = resp.json()
    token["expires_at"] = time.time() + token["expires_in"] - 30
    return token

def refresh_cdse_token(token: dict) -> dict:
    if time.time() >= token.get("expires_at", 0):
        return get_cdse_token()
    return token

token = get_cdse_token()
print("Token CDSE obtenu.")

In [ ]:
# ── Découverte du granule_id par appel OData /Nodes/ ─────────────────
# Le granule_id est le nom du répertoire sous GRANULE/ dans l'arborescence
# SAFE (ex. L2A_T30UYA_A012345_20240115T105259). Il n'est pas disponible
# dans la réponse catalogue — on le récupère via un appel Nodes léger
# (métadonnées uniquement, pas de téléchargement).

def get_tile_crs(tile_id: str) -> CRS:
    """Retourne le CRS UTM d'une tuile Sentinel-2 à partir de son identifiant."""
    # tile_id ex: 30UYA → zone 30 Nord, 31UCR → zone 31 Nord
    zone = int(tile_id[:2])
    return CRS.from_epsg(32600 + zone)   # WGS84 UTM Nord

def get_granule_id(product_id: str, scene_id: str, token: dict) -> str | None:
    """
    Retourne le nom du premier répertoire sous GRANULE/ dans l'arborescence
    SAFE du produit, ou None en cas d'erreur.
    """
    url = (
        f"{ODATA_BASE_DL}/Products({product_id})"
        f"/Nodes({scene_id}.SAFE)"
        f"/Nodes(GRANULE)/Nodes"
    )
    try:
        resp = requests.get(
            url,
            headers={"Authorization": f"Bearer {token['access_token']}"},
            timeout=30,
        )
        resp.raise_for_status()
        nodes = resp.json().get("result", [])
        if nodes:
            return nodes[0]["Name"]
    except Exception as e:
        print(f"  WARN get_granule_id({scene_id}): {e}")
    return None

# Test sur la première scène
row0 = df_dedup.iloc[0]
granule_test = get_granule_id(row0["product_id"], row0["scene_id"], token)
print(f"Test granule_id → {granule_test}")

In [ ]:
# ── Téléchargement SCL et calcul f_valid_aoi ─────────────────────────

def compute_f_valid_aoi(
    scl_path: Path,
    aoi_geom,           # géométries AOI reprojetées dans le CRS de la SCL
    tile_id,
) -> float:
    """
    Calcule la fraction de pixels valides (hors classes SCL_INVALIDES)
    dans l'emprise de l'AOI.
    Retourne NaN si le masque est vide.
    """
    with rasterio.open(scl_path) as src:
        # Reprojeter l'AOI dans le CRS de la SCL si nécessaire
        scl_crs = src.crs or get_tile_crs(tile_id)
        epsg = scl_crs.to_epsg()
        aoi_reproj = aoi_geom.to_crs(epsg)
        shapes = [geom.__geo_interface__ for geom in aoi_reproj.geometry]
        try:
            data, _ = rasterio.mask.mask(src, shapes, crop=True, nodata=0)
        except Exception:
            return float("nan")
        scl = data[0]
        total = np.sum(scl > 0)          # pixels non-nodata
        if total == 0:
            return float("nan")
        invalides = np.sum(np.isin(scl, list(SCL_INVALIDES)))
        return float((total - invalides) / total)


def process_scene(row: pd.Series, token: dict, aoi: gpd.GeoDataFrame) -> tuple:
    """
    Pour une scène donnée :
      1. Découverte du granule_id
      2. Téléchargement de la SCL (skip si déjà présente)
      3. Calcul de f_valid_aoi
    Retourne (scene_id, f_valid_aoi).
    """
    token = refresh_cdse_token(token)
    scene_id   = row["scene_id"]
    product_id = row["product_id"]
    tile_id    = row["tile_id"]

    # Dossier de sortie par tuile
    out_dir = SCL_DIR / tile_id
    out_dir.mkdir(parents=True, exist_ok=True)
    scl_path = out_dir / f"{scene_id}_SCL_60m.jp2"

    if not scl_path.exists():
        # Découverte du granule_id
        granule_id = get_granule_id(product_id, scene_id, token)
        if granule_id is None:
            return (scene_id, float("nan"))

        # Construction de l'URL SCL
        # Le nom du fichier SCL suit le pattern : <tile>_<date>_SCL_60m.jp2
        # ex. T30UYA_20240115T105259_SCL_60m.jp2
        date_str   = scene_id.split("_")[2]          # YYYYMMDDTHHMMSS
        scl_name   = f"T{tile_id}_{date_str}_SCL_60m.jp2"
        url = (
            f"{ODATA_BASE_DL}/Products({product_id})"
            f"/Nodes({scene_id}.SAFE)"
            f"/Nodes(GRANULE)"
            f"/Nodes({granule_id})"
            f"/Nodes(IMG_DATA)"
            f"/Nodes(R60m)"
            f"/Nodes({scl_name})/$value"
        )

        try:
            token = refresh_cdse_token(token)
            resp = requests.get(
                url,
                headers={"Authorization": f"Bearer {token['access_token']}"},
                timeout=120,
                stream=True,
            )
            resp.raise_for_status()
            with open(scl_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 256):
                    f.write(chunk)
        except Exception as e:
            print(f"  WARN téléchargement SCL ({scene_id}): {e}")
            return (scene_id, float("nan"))

    # Calcul f_valid_aoi
    f = compute_f_valid_aoi(scl_path, aoi, tile_id)
    return (scene_id, f)


# ── Boucle parallèle ─────────────────────────────────────────────────
results = {}   # scene_id → f_valid_aoi

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_scene, row, token, aoi): row["scene_id"]
        for _, row in df_dedup.iterrows()
    }
    for i, future in enumerate(as_completed(futures), 1):
        scene_id, f = future.result()
        results[scene_id] = f
        if i % 50 == 0 or i == len(futures):
            valides = sum(1 for v in results.values() if not np.isnan(v) and v >= F_VALID_SEUIL)
            print(f"  [{i}/{len(futures)}] scènes traitées, {valides} scènes retenues (f≥{F_VALID_SEUIL})")

# Mise à jour du DataFrame
df_dedup["f_valid_aoi"] = df_dedup["scene_id"].map(results)
print(f"\nf_valid_aoi calculé pour {df_dedup['f_valid_aoi'].notna().sum()} / {len(df_dedup)} scènes")

In [ ]:
# ── Contrôle qualité f_valid_aoi ─────────────────────────────────────
n_total    = len(df_dedup)
n_nan      = df_dedup["f_valid_aoi"].isna().sum()
n_retenues = (df_dedup["f_valid_aoi"] >= F_VALID_SEUIL).sum()
n_exclues  = (df_dedup["f_valid_aoi"] < F_VALID_SEUIL).sum()

print(f"Scènes totales       : {n_total}")
print(f"f_valid_aoi NaN      : {n_nan}  (erreurs téléchargement / masque vide)")
print(f"Retenues (≥ {F_VALID_SEUIL})   : {n_retenues}")
print(f"Exclues  (< {F_VALID_SEUIL})   : {n_exclues}")
print()
print("Distribution f_valid_aoi :")
print(df_dedup["f_valid_aoi"].describe().round(3).to_string())

# ── Mise à jour du Parquet ────────────────────────────────────────────
df_dedup.to_parquet(PARQUET_PATH, index=False)
print(f"\nParquet mis à jour : {PARQUET_PATH}")

### 3.2 — Téléchargement des bandes spectrales et calcul des indices

Téléchargement des bandes B02, B04, B05, B06, B07, B08, B11 pour les
scènes retenues (`f_valid_aoi ≥ 0.01`). Les bandes à 20 m (B05, B06,
B07, B11) sont resamplées à 10 m (bilinéaire) pour homogénéiser la
résolution avant le calcul des indices.

Quatre indices spectraux sont calculés par scène sur l'emprise de l'AOI :

| Indice | Formule | Intérêt agronomique |
|--------|---------|---------------------|
| NDVI | (B08 − B04) / (B08 + B04) | Vigueur végétale, phénologie |
| EVI  | 2.5 × (B08 − B04) / (B08 + 6×B04 − 7.5×B02 + 1) | Vigueur sans saturation |
| NDWI | (B08 − B11) / (B08 + B11) | Teneur en eau, stades phénologiques |
| NDRE | (B08 − B05) / (B08 + B05) | Discrimination cultures avant saturation NDVI |

Les bandes et indices sont stockés en GeoTIFF dans
`data/raw/s2/bands/<tile_id>/` et `data/raw/s2/indices/<tile_id>/`
(non versionnés). Les fichiers sources `.jp2` sont supprimés dans une
cellule dédiée, exécutée manuellement après validation des GeoTIFF produits.

In [ ]:
# ── Constantes section 3.2 ────────────────────────────────────────────
BANDS_DIR   = PROJECT_ROOT / "data" / "raw" / "s2" / "bands"
INDICES_DIR = PROJECT_ROOT / "data" / "raw" / "s2" / "indices"

# Bandes à télécharger : (nom_bande, répertoire_SAFE, suffixe_résolution)
BANDES = [
    ("B02", "R10m", "10m"),
    ("B04", "R10m", "10m"),
    ("B05", "R20m", "20m"),
    ("B06", "R20m", "20m"),
    ("B07", "R20m", "20m"),
    ("B08", "R10m", "10m"),
    ("B11", "R20m", "20m"),
]

BANDS_DIR.mkdir(parents=True, exist_ok=True)
INDICES_DIR.mkdir(parents=True, exist_ok=True)
print(f"BANDS_DIR   : {BANDS_DIR}")
print(f"INDICES_DIR : {INDICES_DIR}")

In [ ]:
def download_band(
    product_id: str,
    scene_id: str,
    granule_id: str,
    band: str,
    safe_dir: str,
    res_suffix: str,
    tile_id: str,
    date_str: str,
    token: dict,
) -> Path | None:
    """Télécharge une bande jp2 dans BANDS_DIR/<tile_id>/. Retourne le chemin."""
    out_dir = BANDS_DIR / tile_id
    out_dir.mkdir(parents=True, exist_ok=True)
    fname    = f"T{tile_id}_{date_str}_{band}_{res_suffix}.jp2"
    out_path = out_dir / fname

    if out_path.exists():
        return out_path

    url = (
        f"{ODATA_BASE_DL}/Products({product_id})"
        f"/Nodes({scene_id}.SAFE)"
        f"/Nodes(GRANULE)"
        f"/Nodes({granule_id})"
        f"/Nodes(IMG_DATA)"
        f"/Nodes({safe_dir})"
        f"/Nodes({fname})/$value"
    )
    try:
        token = refresh_cdse_token(token)
        resp = requests.get(
            url,
            headers={"Authorization": f"Bearer {token['access_token']}"},
            timeout=180,
            stream=True,
        )
        resp.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 256):
                f.write(chunk)
        return out_path
    except Exception as e:
        print(f"  WARN download_band({scene_id}, {band}): {e}")
        return None


def resample_to_20m(
    src_path: Path,
    ref_transform,
    ref_crs_wkt: str,
    ref_shape: tuple,
) -> np.ndarray:
    """Relit une bande jp2 et la resamle à la grille 20m de référence (array-to-array).

    CORRECTIF 1 (diagnostic nb05, artefact bimodal dist_classe) :
    les JP2 L2A codent en 0 les pixels hors fauchée satellite (les coins
    du carré MGRS non couverts par le passage incliné du satellite,
    distincts de l'emprise carrée de la tuile). Sans src_nodata=0, GDAL
    traite ces 0 comme une réflectance valide et les mélange par
    interpolation bilinéaire aux pixels voisins réels, produisant une
    frontière nette et rectiligne à l'intérieur de la tuile (le bord de
    fauchée), décorrélée du footprint de tuile — exactement le pattern
    observé dans les composites mensuels. dst_nodata=np.nan propage
    proprement l'absence de donnée jusqu'au masque SCL et au composite.

    CORRECTIF 2 (diagnostic 3.4, parcelles orphelines 31UCQ/31UCR) :
    quand le JP2 source n'expose pas son CRS dans ses métadonnées (driver
    JP2OpenJPEG sur Windows), l'ancien fallback était codé en dur sur
    EPSG:32630 (zone UTM 30N) — correct par coïncidence pour 30UYA/30UYV,
    mais faux pour 31UCQ/31UCR (zone 31N), décalant les données d'un
    fuseau entier lors de la reprojection et produisant un résultat
    intégralement nodata (149/149 et 145/145 scènes affectées, 100 %
    systématique — le JP2OpenJPEG de cet environnement n'expose jamais le
    CRS pour ces deux tuiles). Le fallback réutilise maintenant ref_crs_wkt,
    déjà résolu correctement pour la tuile via get_tile_crs() en amont
    (process_scene_s32) — toutes les bandes d'une même scène partagent le
    même CRS natif, pas besoin de le redeviner bande par bande.
    """
    with rasterio.open(src_path) as src:
        raw = src.read(1).astype(np.float32)
        src_crs_wkt = (src.crs or CRS.from_wkt(ref_crs_wkt)).to_wkt()
        src_transform = src.transform
    data = np.full(ref_shape, np.nan, dtype=np.float32)
    rasterio.warp.reproject(
        source=raw,
        destination=data,
        src_transform=src_transform,
        src_crs=src_crs_wkt,
        dst_transform=ref_transform,
        dst_crs=ref_crs_wkt,
        resampling=Resampling.bilinear,
        src_nodata=0,        # ← correctif : 0 = hors fauchée dans les JP2 L2A
        dst_nodata=np.nan,   # ← correctif : propage le nodata (pas de blending 0/valeur réelle)
    )
    return data


def compute_indices(bands: dict) -> dict:
    """Calcule NDVI, EVI, NDWI, NDRE à partir du dict de tableaux numpy."""
    eps = 1e-10  # évite la division par zéro
    b02 = bands["B02"].astype(np.float32) / 10000
    b04 = bands["B04"].astype(np.float32) / 10000
    b05 = bands["B05"].astype(np.float32) / 10000
    b08 = bands["B08"].astype(np.float32) / 10000
    b11 = bands["B11"].astype(np.float32) / 10000

    ndvi = (b08 - b04) / (b08 + b04 + eps)
    ndwi = (b08 - b11) / (b08 + b11 + eps)
    ndre = (b08 - b05) / (b08 + b05 + eps)

    evi_denom = b08 + 6 * b04 - 7.5 * b02 + 1
    evi_denom = np.where(np.abs(evi_denom) < 0.001, 0.001, evi_denom)
    evi  = 2.5 * (b08 - b04) / evi_denom

    # Clamp dans [-1, 1]
    return {
        "NDVI": np.clip(ndvi, -1, 1),
        "EVI":  np.clip(evi,  -2, 2),
        "NDWI": np.clip(ndwi, -1, 1),
        "NDRE": np.clip(ndre, -1, 1),
    }


def save_geotiff(
    data: np.ndarray,
    out_path: Path,
    transform,
    crs,
    nodata: float = -9999.0,
) -> None:
    """Sauvegarde un tableau numpy en GeoTIFF Float32 compressé."""
    with rasterio.open(
        out_path, "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype="float32",
        crs=crs,
        transform=transform,
        nodata=nodata,
        compress="deflate",
        tiled=True,
        blockxsize=256,
        blockysize=256,
    ) as dst:
        data_out = np.where(np.isnan(data), nodata, data).astype(np.float32)
        dst.write(data_out, 1)

print("Fonctions définies.")


In [ ]:
# ── Fonction de traitement d'une scène complète ───────────────────────
def process_scene_s32(row: pd.Series) -> tuple:
    """
    Pour une scène retenue :
      1. Téléchargement des 7 bandes
      2. Lecture + crop AOI + resampling 20m
      3. Calcul indices NDVI/EVI/NDWI/NDRE
      4. Sauvegarde bandes et indices en GeoTIFF
    Retourne (scene_id, "OK") ou (scene_id, "ERREUR <msg>").
    """
    scene_id   = row["scene_id"]
    product_id = row["product_id"]
    tile_id    = row["tile_id"]
    date_str   = scene_id.split("_")[2]

    # Skip si déjà traité
    idx_dir   = INDICES_DIR / tile_id
    ndvi_path = idx_dir / f"{scene_id}_NDVI.tif"
    if ndvi_path.exists():
        return (scene_id, "SKIP")

    # Token local au thread
    t = get_cdse_token()

    # Découverte granule_id
    granule_id = get_granule_id(product_id, scene_id, t)
    if granule_id is None:
        return (scene_id, "ERREUR granule_id introuvable")

    # Téléchargement des 7 bandes
    band_paths = {}
    for band, safe_dir, res_suffix in BANDES:
        t = refresh_cdse_token(t)
        p = download_band(
            product_id, scene_id, granule_id,
            band, safe_dir, res_suffix,
            tile_id, date_str, t,
        )
        if p is None:
            return (scene_id, f"ERREUR téléchargement {band}")
        band_paths[band] = p

    # Lecture B05 (20m natif) — crop AOI, définit la grille de référence 20m
    try:
        with rasterio.open(band_paths["B05"]) as ref:
            ref_crs    = ref.crs
            _crs = ref_crs or ProjCRS.from_epsg(32600 + int(tile_id[:2]))
            aoi_reproj = aoi.to_crs(_crs.to_epsg())
            shapes     = [g.__geo_interface__ for g in aoi_reproj.geometry]
            b05_raw, transform = rasterio.mask.mask(ref, shapes, crop=True, nodata=0)
            ref_shape  = b05_raw.shape[1:]
            ref_crs    = _crs
            del b05_raw

        # ── Masque SCL sur l’emprise AOI ────────────────────────────
        scl_path_s32 = SCL_DIR / tile_id / f"{scene_id}_SCL_60m.jp2"
        scl_reproj = np.zeros(ref_shape, dtype=np.uint8)
        if scl_path_s32.exists():
            with rasterio.open(scl_path_s32) as scl_src:
                scl_raw = scl_src.read(1)
                scl_crs_wkt = (scl_src.crs or get_tile_crs(tile_id)).to_wkt()
                scl_transform = scl_src.transform
            rasterio.warp.reproject(
                source=scl_raw,
                destination=scl_reproj,
                src_transform=scl_transform,
                src_crs=scl_crs_wkt,
                dst_transform=transform,
                dst_crs=ref_crs.to_wkt(),
                resampling=Resampling.nearest,
            )
            del scl_raw
        else:
            print(f"  WARN SCL absente pour {scene_id} — masque non appliqué")
        masque_invalide = np.isin(scl_reproj, list(SCL_INVALIDES))
        del scl_reproj

        # Lecture + resampling 20m uniforme pour toutes les bandes
        bands = {}
        for band, _, _ in BANDES:
            bands[band] = resample_to_20m(
                band_paths[band], transform, ref_crs.to_wkt(), ref_shape
            )

        # Appliquer le masque SCL à toutes les bandes
        for band in bands:
            bands[band] = bands[band].astype(np.float32)
            bands[band][masque_invalide] = np.nan
        del masque_invalide

        # Sauvegarde bandes
        band_out_dir = BANDS_DIR / tile_id
        band_out_dir.mkdir(parents=True, exist_ok=True)
        for band, arr in bands.items():
            save_geotiff(arr, band_out_dir / f"{scene_id}_{band}.tif", transform, ref_crs)

        # Calcul et sauvegarde indices
        idx_dir.mkdir(parents=True, exist_ok=True)
        indices = compute_indices(bands)
        for name, arr in indices.items():
            save_geotiff(arr, idx_dir / f"{scene_id}_{name}.tif", transform, ref_crs)

    except Exception as e:
        return (scene_id, f"ERREUR {e}")
    finally:
        gc.collect()

    return (scene_id, "OK")


# ── Boucle séquentielle ──────────────────────────────────────────────
df_retenues = df_dedup[df_dedup["f_valid_aoi"] >= F_VALID_SEUIL].reset_index(drop=True)
print(f"Scènes à traiter : {len(df_retenues)}")

erreurs = []
n_ok    = 0
n_skip  = 0

for i, (_, row) in enumerate(df_retenues.iterrows(), 1):
    scene_id, status = process_scene_s32(row)
    if status == "OK":
        n_ok += 1
    elif status == "SKIP":
        n_skip += 1
    else:
        erreurs.append((scene_id, status))
    print(f"  [{i}/{len(df_retenues)}] OK:{n_ok} SKIP:{n_skip} ERR:{len(erreurs)} — {status}")

print(f"\nTerminé. OK:{n_ok}  SKIP:{n_skip}  Erreurs:{len(erreurs)}")
if erreurs:
    for scene, msg in erreurs:
        print(f"  {scene} — {msg}")

### 3.2 bis — Contrôle qualité des correctifs `resample_to_20m`

Deux vérifications sur les bandes (et indices) fraîchement produits :

1. **Zéros résiduels** — qu'aucun pixel à exactement `0` ne subsiste
   dans l'emprise AOI (hors bordure de fauchée non masquée). Un `0`
   résiduel indiquerait que le correctif `src_nodata=0` n'a pas été
   pris en compte pour cette scène (ex. fichier généré avant le
   correctif et non régénéré — supprimer et relancer 3.2 pour cette
   scène si besoin).

2. **Fichiers entièrement nodata** — qu'aucun `.tif` de bande ou
   d'indice n'est composé à 100 % de la valeur nodata. Ce cas ne
   déclenche aucune erreur silencieuse en aval (les stats zonales
   et composites l'ignorent proprement via `nanmedian`), mais révèle
   un échec de reprojection total pour la scène — historiquement
   causé par le fallback CRS de `resample_to_20m`, codé en dur sur
   la zone 30N avant correctif, qui décalait d'un fuseau entier les
   tuiles en zone 31N (31UCQ/31UCR) chaque fois que le JP2 source
   n'exposait pas son CRS. Diagnostiqué en 3.4 (2 278 parcelles
   « orphelines », jamais aucune valeur S2 malgré une couverture
   nominale par 2 tuiles) — voir `methode.md` pour le détail de
   l'enquête.

In [ ]:
# ── QC 3.2 bis : complétude, zéros résiduels, fichiers entièrement nodata ──
# Piloté par df_retenues (pas par scan de dossier) : on sait a priori ce
# qui DOIT exister pour chaque scène retenue, donc un fichier manquant
# est détecté explicitement, pas juste absent du scan.
BANDES_ATTENDUES = [b for b, _, _ in BANDES]           # 7 bandes
INDICES_ATTENDUS = ["NDVI", "EVI", "NDWI", "NDRE"]      # 4 indices

problemes = []   # (scene_id, tile_id, type_pb, detail)

for _, row in df_retenues.iterrows():
    scene_id = row["scene_id"]
    tile_id  = row["tile_id"]
    band_dir = BANDS_DIR / tile_id
    idx_dir  = INDICES_DIR / tile_id

    fichiers_attendus = (
        [(band_dir / f"{scene_id}_{b}.tif", f"bande {b}") for b in BANDES_ATTENDUES]
        + [(idx_dir / f"{scene_id}_{i}.tif", f"indice {i}") for i in INDICES_ATTENDUS]
    )

    manquants = [label for path, label in fichiers_attendus if not path.exists()]
    if manquants:
        problemes.append((scene_id, tile_id, "MANQUANT", manquants))
        continue  # pas la peine de vérifier la validité si déjà incomplet

    vides = []
    zeros_residuels = []
    for path, label in fichiers_attendus:
        with rasterio.open(path) as src:
            arr = src.read(1)
            if np.all(arr == src.nodata):
                vides.append(label)
            elif "indice" not in label:  # zéros résiduels : bandes uniquement (attendu absent sur indices)
                n_zero = int(np.sum(arr == 0))
                if n_zero > 0:
                    zeros_residuels.append(f"{label} ({n_zero:,} px)")

    if vides:
        problemes.append((scene_id, tile_id, "VIDE", vides))
    if zeros_residuels:
        problemes.append((scene_id, tile_id, "ZEROS_RESIDUELS", zeros_residuels))

# ── Rapport ────────────────────────────────────────────────────────────
print(f"{len(df_retenues)} scènes vérifiées ({len(BANDES_ATTENDUES) + len(INDICES_ATTENDUS)} fichiers/scène attendus).")

if problemes:
    print(f"\n⚠ {len(problemes)} problème(s) détecté(s) — SUPPRESSION DES JP2 À NE PAS LANCER :")
    for scene_id, tile_id, type_pb, detail in problemes[:30]:
        print(f"  [{type_pb}] {scene_id} ({tile_id}) — {detail}")
    if len(problemes) > 30:
        print(f"  ... et {len(problemes) - 30} autres")

    tuiles_touchees = sorted({p[1] for p in problemes})
    par_type = {}
    for p in problemes:
        par_type[p[2]] = par_type.get(p[2], 0) + 1
    print(f"\nTuiles concernées : {tuiles_touchees}")
    print(f"Répartition : {par_type}")
else:
    print("✅ Toutes les scènes ont leurs 11 fichiers (7 bandes + 4 indices), tous valides, aucun zéro résiduel.")
    print("   Suppression des .jp2 (cellule suivante) : OK pour lancer.")


In [ ]:
# ── Suppression des fichiers .jp2 après validation des GeoTIFF ────────
# EXÉCUTER MANUELLEMENT après avoir vérifié que tous les indices
# et bandes GeoTIFF ont bien été produits.

jp2_files = list(BANDS_DIR.rglob("*.jp2"))
print(f"Fichiers .jp2 à supprimer : {len(jp2_files)}")

# Décommenter pour exécuter la suppression :
# for f in jp2_files:
#     f.unlink()
# print("Suppression terminée.")


### 3.2 ter — Grille AOI de référence et mois complets

Grille commune en EPSG:2154 à 20 m, utilisée par tous les traitements
raster en aval (contrôle qualité de couverture en 3.2 quater,
composites de 3.3, stats zonales de 3.4/3.5) — garantit l'alignement
pixel entre toutes les étapes.

Détermine aussi la liste des mois pour lesquels toutes les scènes
retenues (`f_valid_aoi ≥ F_VALID_SEUIL`) ont déjà leurs indices
produits en 3.2 — condition nécessaire à tout traitement raster en
aval sur ce mois.


In [ ]:
# ── Grille AOI de référence en EPSG:2154 à 20 m ──────────────────────
RES       = 20        # résolution cible en mètres
CRS_CIBLE     = "EPSG:2154"               # conservé pour geopandas / to_crs
CRS_CIBLE_WKT = ProjCRS.from_epsg(2154).to_wkt()  # pour rasterio.warp (contournement PROJ Windows)

aoi_2154  = aoi.to_crs(CRS_CIBLE)
bounds    = aoi_2154.total_bounds          # (xmin, ymin, xmax, ymax)

# Arrondir l'emprise aux 20 m
xmin = np.floor(bounds[0] / RES) * RES
ymin = np.floor(bounds[1] / RES) * RES
xmax = np.ceil(bounds[2]  / RES) * RES
ymax = np.ceil(bounds[3]  / RES) * RES

AOI_WIDTH     = int((xmax - xmin) / RES)
AOI_HEIGHT    = int((ymax - ymin) / RES)
AOI_TRANSFORM = from_bounds(xmin, ymin, xmax, ymax, AOI_WIDTH, AOI_HEIGHT)

print(f"Grille AOI : {AOI_WIDTH} × {AOI_HEIGHT} pixels à {RES} m (EPSG:2154)")
print(f"Emprise    : {xmin:.0f} {ymin:.0f} → {xmax:.0f} {ymax:.0f}")
print(f"Surface    : {AOI_WIDTH * AOI_HEIGHT * RES**2 / 1e6:.1f} km²")


In [ ]:
def reproject_to_aoi(src_path: Path) -> np.ndarray:
    """
    Reprojette un GeoTIFF source sur la grille AOI 20 m EPSG:2154.
    Retourne un tableau float32 (AOI_HEIGHT, AOI_WIDTH) avec NaN sur nodata.
    """
    with rasterio.open(src_path) as src:
        nodata = src.nodata if src.nodata is not None else -9999.0
        dst = np.full((AOI_HEIGHT, AOI_WIDTH), np.nan, dtype=np.float32)
        rasterio.warp.reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=AOI_TRANSFORM,
            dst_crs=CRS_CIBLE_WKT,
            resampling=Resampling.bilinear,
            src_nodata=nodata,
            dst_nodata=np.nan,
        )
    return dst


In [ ]:
# ── Détermination des mois complets (couverture bandes/indices) ──────
import warnings
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

df_retenues = df_dedup[df_dedup["f_valid_aoi"] >= F_VALID_SEUIL].copy()
df_retenues["mois"]  = pd.to_datetime(df_retenues["date"]).dt.to_period("M").astype(str)
df_retenues["date8"] = pd.to_datetime(df_retenues["date"]).dt.strftime("%Y%m%d")

mois_list = sorted(df_retenues["mois"].unique())

mois_complets = []
for mois in mois_list:
    df_mois = df_retenues[df_retenues["mois"] == mois]
    tous_presents = all(
        (INDICES_DIR / sid.split("_")[5][1:] / f"{sid}_NDVI.tif").exists()
        for sid in df_mois["scene_id"]
    )
    if tous_presents:
        mois_complets.append(mois)

print(f"Mois complets ({len(mois_complets)}/{len(mois_list)}) : {mois_complets}")
print(f"Mois incomplets (à traiter après fin de 3.2) : "
      f"{[m for m in mois_list if m not in mois_complets]}")


### 3.2 quater — Contrôle qualité couverture temporelle

Pour chaque mois, calcule le nombre de dates valides par pixel AOI et le
pourcentage de pixels n'ayant reçu aucune date valide. Un mois avec un
fort pourcentage de pixels à 0 date valide (typiquement l'hiver, en
raison de la nébulosité) produira des trous NaN étendus dans les
composites — pas un bug, mais un déficit de couverture à anticiper
pour S3/S4.

Le raster de complétude (`n_valid_mois`) est sauvegardé par mois dans
`data/completude/` pour être réutilisé en 3.5 (stats zonales de
complétude par parcelle). Idempotent : un mois n'est recalculé que si
au moins un fichier indice source est plus récent que le raster de
complétude déjà présent.

Indépendant des composites de 3.3 — ne s'appuie que sur les indices
par scène produits en 3.2 et sur la grille/liste de mois de 3.2 ter.


In [ ]:
# ── QC : couverture temporelle par mois — pixels sans date valide ────
import time

VARIABLE_QC = "NDVI"   # une seule variable suffit : le masque de
                        # validité (SCL + hors-fauchée) est identique
                        # pour toutes les variables d'une même scène

COMPLETUDE_DIR = PROJECT_ROOT / "data" / "completude"
COMPLETUDE_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_COMPLETUDE = {
    "driver": "GTiff", "dtype": "int16", "count": 1,
    "height": AOI_HEIGHT, "width": AOI_WIDTH,
    "crs": CRS_CIBLE_WKT, "transform": AOI_TRANSFORM,
    "compress": "deflate", "tiled": True,
    "blockxsize": 256, "blockysize": 256,
    # pas de nodata : 0 est une valeur valide ("0 date"), pas une absence
}


def completude_a_jour(df_m: pd.DataFrame, out_path: Path) -> bool:
    """Vérifie que le raster de complétude est plus récent que tous
    les fichiers indices sources du mois (sinon, à recalculer)."""
    if not out_path.exists():
        return False
    mtime_out = out_path.stat().st_mtime
    for _, row in df_m.iterrows():
        tile_id = row["tile_id"]
        src_path = INDICES_DIR / tile_id / f"{row['scene_id']}_{VARIABLE_QC}.tif"
        if src_path.exists() and src_path.stat().st_mtime > mtime_out:
            return False
    return True


resultats_qc = []
t_debut_global = time.time()
temps_par_mois = []

for i_mois, mois in enumerate(mois_complets, 1):
    t0_mois = time.time()
    out_path = COMPLETUDE_DIR / f"{mois}_n_valid.tif"

    df_m = df_retenues[df_retenues["mois"] == mois]
    scene_ids_par_date = df_m.groupby("date8")["scene_id"].apply(list).to_dict()
    n_dates_mois = len(scene_ids_par_date)

    # ── Skip si le raster de complétude est déjà à jour ───────────────
    if completude_a_jour(df_m, out_path):
        with rasterio.open(out_path) as src:
            n_valid_mois = src.read(1)
        pct_zero = float(np.mean(n_valid_mois == 0) * 100)
        print(f"\n[{i_mois}/{len(mois_complets)}] {mois} — "
              f"SKIP (à jour, {n_dates_mois} dates) — {pct_zero:.1f} % à 0 date")
        resultats_qc.append({
            "mois": mois,
            "n_dates_retenues": n_dates_mois,
            "pct_pixels_0_date": round(pct_zero, 1),
        })
        temps_par_mois.append(0.0)
        continue

    # ── Sinon, calcul complet ──────────────────────────────────────────
    if temps_par_mois and any(t > 0 for t in temps_par_mois):
        temps_moyen_mois = (
            sum(t for t in temps_par_mois if t > 0)
            / sum(1 for t in temps_par_mois if t > 0)
        )
        mois_restants = len(mois_complets) - i_mois + 1
        eta_str = f" — ETA global : {temps_moyen_mois * mois_restants / 60:.1f} min"
    else:
        eta_str = ""

    print(f"\n[{i_mois}/{len(mois_complets)}] {mois} — {n_dates_mois} dates à traiter{eta_str}")

    n_valid_mois = np.zeros((AOI_HEIGHT, AOI_WIDTH), dtype=np.int16)

    for i_date, (date_str, scene_ids) in enumerate(scene_ids_par_date.items(), 1):
        t0_date = time.time()
        tuile_arrays = []
        for scene_id in scene_ids:
            tile_id  = scene_id.split("_")[5][1:]
            src_path = INDICES_DIR / tile_id / f"{scene_id}_{VARIABLE_QC}.tif"
            if src_path.exists():
                tuile_arrays.append(reproject_to_aoi(src_path))
        if tuile_arrays:
            jour = (
                tuile_arrays[0] if len(tuile_arrays) == 1
                else np.nanmedian(np.stack(tuile_arrays, axis=0), axis=0)
            )
            n_valid_mois += (~np.isnan(jour)).astype(np.int16)

        dt_date = time.time() - t0_date
        print(f"    [{i_date}/{n_dates_mois}] {date_str} — "
              f"{len(tuile_arrays)}/{len(scene_ids)} tuiles — {dt_date:.1f}s")

    dt_mois = time.time() - t0_mois
    temps_par_mois.append(dt_mois)

    pct_zero = float(np.mean(n_valid_mois == 0) * 100)
    resultats_qc.append({
        "mois": mois,
        "n_dates_retenues": n_dates_mois,
        "pct_pixels_0_date": round(pct_zero, 1),
    })
    print(f"  → {mois} terminé en {dt_mois:.1f}s — "
          f"{pct_zero:.1f} % des pixels AOI sans aucune date valide")

    # ── Sauvegarde du raster de complétude pour réutilisation en 3.5 ──
    with rasterio.open(out_path, "w", **PROFILE_COMPLETUDE) as dst:
        dst.write(n_valid_mois.astype("int16"), 1)
    print(f"  → raster de complétude sauvegardé : {out_path.relative_to(PROJECT_ROOT)}")

dt_total = time.time() - t_debut_global
print(f"\n{'='*60}")
print(f"Terminé en {dt_total/60:.1f} min "
      f"({sum(1 for t in temps_par_mois if t == 0)} mois skippés)")

df_qc = pd.DataFrame(resultats_qc)
print(df_qc.to_string(index=False))

print("\nMois à risque (> 20 % de pixels sans date valide) :")
print(df_qc[df_qc["pct_pixels_0_date"] > 20].to_string(index=False))

df_qc.to_json(PROJECT_ROOT / "data" / "qc_couverture_mensuelle.json",
              orient="records", indent=2)
print(f"\nRésumé QC sauvegardé : data/qc_couverture_mensuelle.json")


In [ ]:
# ── Bar chart : % pixels sans date valide par mois (mise à jour possible en cours de run) ──
df_qc_partiel = pd.DataFrame(resultats_qc)

SEUIL_RISQUE = 20  # % — à ajuster selon la distribution observée

fig, ax = plt.subplots(figsize=(12, 5))
couleurs = ["crimson" if p > SEUIL_RISQUE else "steelblue"
            for p in df_qc_partiel["pct_pixels_0_date"]]
bars = ax.bar(df_qc_partiel["mois"], df_qc_partiel["pct_pixels_0_date"], color=couleurs)

ax.axhline(SEUIL_RISQUE, color="crimson", linestyle="--", linewidth=1,
           label=f"Seuil de risque ({SEUIL_RISQUE} %)")
ax.set_ylabel("% pixels AOI sans date valide")
ax.set_title("Couverture temporelle par mois — S2 composites")
ax.set_xticklabels(df_qc_partiel["mois"], rotation=45, ha="right")

# Annoter chaque barre avec le nb de dates retenues
for bar, n_dates in zip(bars, df_qc_partiel["n_dates_retenues"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{n_dates}d", ha="center", fontsize=8)

ax.legend()
plt.tight_layout()
plt.show()

print(f"\n{len(df_qc_partiel)}/{len(mois_complets)} mois traités jusqu'ici")


### 3.3 — Composite mensuel

Construction du composite mensuel AOI en deux étapes, pour chaque
bande et indice (11 variables), sur la grille AOI et la liste des
mois complets définies en 3.2 ter.

**Étape 1 — image journalière** : pour chaque date d'acquisition du
mois, tous les GeoTIFF disponibles (toutes tuiles confondues) sont
reprojetés sur la grille AOI commune en EPSG:2154 à 20 m (bilinéaire),
puis agrégés par médiane pixel à pixel. Chaque pixel reçoit ainsi la
médiane des valeurs valides issues de toutes les tuiles qui le couvrent
ce jour-là (1 à 4 selon les recouvrements).

**Étape 2 — composite mensuel** : médiane pixel à pixel de toutes les
images journalières valides (`f_valid_aoi ≥ 0.01`) du mois. Un pixel
sans aucune acquisition valide dans le mois reçoit la valeur nodata
(`-9999`).

La médiane est robuste aux nuages résiduels non détectés par la SCL
et aux outliers radiométriques ponctuels.

**Structure de sortie** :
`data/raw/s2/composites/<YYYY-MM>/<variable>.tif` — un GeoTIFF AOI
par mois × variable.


In [ ]:
# ── Constantes 3.3 ────────────────────────────────────────────────────
VARIABLES = ["B02", "B04", "B05", "B06", "B07", "B08", "B11",
             "NDVI", "EVI", "NDWI", "NDRE"]
COMPOSITES_DIR = PROJECT_ROOT / "data" / "raw" / "s2" / "composites"
COMPOSITES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def compute_monthly_composite(
    mois: str,
    scene_ids_par_date: dict,   # {date_str: [scene_id, ...]} pour ce mois
    variable: str,
) -> tuple:
    """
    Calcule le composite mensuel AOI pour un mois × variable.
    Étape 1 : image journalière = médiane des tuiles disponibles ce jour.
    Étape 2 : composite mensuel = médiane des images journalières.
    Retourne (mois, variable, "OK"|"SKIP"|"VIDE"|"ERR <msg>").
    """
    out_dir  = COMPOSITES_DIR / mois
    out_path = out_dir / f"{variable}.tif"
    if out_path.exists():
        return (mois, variable, "SKIP")

    src_dir_fn = lambda tile_id: (
        INDICES_DIR / tile_id if variable in ["NDVI", "EVI", "NDWI", "NDRE"]
        else BANDS_DIR / tile_id
    )

    try:
        daily_images = []

# Étape 1 — image journalière par date
        for date_str, scene_ids in scene_ids_par_date.items():
            tuile_arrays = []
            for scene_id in scene_ids:
                tile_id  = scene_id.split("_")[5][1:]
                src_path = src_dir_fn(tile_id) / f"{scene_id}_{variable}.tif"
                if src_path.exists():
                    arr = reproject_to_aoi(src_path)
                    tuile_arrays.append(arr)

            if tuile_arrays:
                if len(tuile_arrays) == 1:
                    daily_images.append(tuile_arrays[0])
                else:
                    daily_images.append(
                        np.nanmedian(np.stack(tuile_arrays, axis=0), axis=0)
                    )
            del tuile_arrays   # ← gc.collect() retiré ici

        if not daily_images:
            return (mois, variable, "VIDE")

        # Étape 2 — composite mensuel
        stack     = np.stack(daily_images, axis=0)
        composite = np.nanmedian(stack, axis=0).astype(np.float32)
        composite[np.isnan(composite)] = -9999.0
        del daily_images, stack
        gc.collect()

        # Sauvegarde
        out_dir.mkdir(parents=True, exist_ok=True)
        with rasterio.open(
            out_path, "w",
            driver="GTiff",
            height=AOI_HEIGHT, width=AOI_WIDTH,
            count=1, dtype="float32",
            crs=CRS_CIBLE_WKT,
            transform=AOI_TRANSFORM,
            nodata=-9999.0,
            compress="deflate",
            tiled=True, blockxsize=256, blockysize=256,
        ) as dst:
            dst.write(composite, 1)

        del composite
        gc.collect()
        return (mois, variable, "OK")

    except Exception as e:
        return (mois, variable, f"ERR {e}")


In [ ]:
# ── Boucle principale ─────────────────────────────────────────────────
total    = len(mois_complets) * len(VARIABLES)
n_ok = n_skip = n_vide = n_err = 0
done = 0

print(f"\nComposites à produire : {total}")

for mois in mois_complets:
    df_mois = df_retenues[df_retenues["mois"] == mois]

    scene_ids_par_date = (
        df_mois.groupby("date8")["scene_id"]
        .apply(list)
        .to_dict()
    )

    print(f"\n{mois} — {len(scene_ids_par_date)} dates")

    for variable in VARIABLES:
        _, _, status = compute_monthly_composite(mois, scene_ids_par_date, variable)
        if status == "OK":       n_ok   += 1
        elif status == "SKIP":   n_skip += 1
        elif status == "VIDE":   n_vide += 1
        else:                    n_err  += 1
        done += 1
        print(f"    [{done}/{total}] {mois} / {variable:<5} — {status}")

    gc.collect()

print(f"\nTerminé. OK:{n_ok}  SKIP:{n_skip}  VIDE:{n_vide}  ERR:{n_err}")


### 3.4 — Agrégation zonale et chargement PostGIS

Extraction des statistiques zonales de chaque composite mensuel pour
les 80 689 parcelles RPG de l'AOI, et chargement dans la table
`derived.s2_parcelles_monthly`.

**Statistiques** : pour chaque combinaison parcelle × mois × variable,
quatre statistiques sont calculées sur les pixels valides du composite :
**mean**, **std**, **p10**, **p90**. La moyenne et l'écart-type capturent
la tendance centrale et l'hétérogénéité intra-parcelle ; les percentiles
10 et 90 enrichissent le feature set pour la classification (S3) sans
surcoût significatif.

**Masquage** : les pixels nodata du composite (`-9999`) sont exclus du
calcul. Les parcelles sans aucun pixel valide pour un mois × variable
donné reçoivent des valeurs `NaN` (insérées comme `NULL` en PostGIS).

**Table cible** : `derived.s2_parcelles_monthly`, clé primaire composite
`(id_parcel, mois, variable)`. Volumétrie cible : ~14,2 M de lignes
(80 689 parcelles × 11 variables × 16 mois). Les insertions utilisent
`INSERT ... ON CONFLICT DO NOTHING` pour permettre les relances
partielles sans duplication — si un composite est régénéré, supprimer
manuellement les lignes du mois × variable concerné avant relance.

**Feature set résultant** : 704 features par parcelle
(11 variables × 4 stats × 16 mois), directement exploitables en wide
format pour l'entraînement du modèle de classification (S3).

**Géométries parcellaires** : lues depuis `derived.rpg_parcelles_aoi`
(PostGIS) en une seule requête GeoDataFrame, projetées en EPSG:2154 —
même CRS que les composites, ce qui garantit l'alignement spatial pour
`rasterio.mask.mask`.

In [ ]:
# ── Connexion PostGIS et création de la table cible ──────────────────
PG_PARAMS = {
    "host":     os.getenv("PG_HOST", "localhost"),
    "port":     int(os.getenv("PG_PORT", 5432)),
    "dbname":   os.getenv("PG_DBNAME", "seinecrops"),
    "user":     os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD"),
}

conn = psycopg2.connect(**PG_PARAMS)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS derived.s2_parcelles_monthly (
            id_parcel  TEXT  NOT NULL,
            mois       TEXT  NOT NULL,
            variable   TEXT  NOT NULL,
            mean       REAL,
            std        REAL,
            p10        REAL,
            p90        REAL,
            PRIMARY KEY (id_parcel, mois, variable)
        );
    """)

print("Table derived.s2_parcelles_monthly prête.")

In [ ]:
# ── Chargement des parcelles et construction du raster de labels ─────

# Identifier la colonne géométrique (wkb_geometry ou geom selon PGDUMP)
with conn.cursor() as cur:
    cur.execute("""
        SELECT f_geometry_column
        FROM geometry_columns
        WHERE f_table_schema = 'derived'
          AND f_table_name   = 'rpg_parcelles_aoi'
    """)
    GEOM_COL = cur.fetchone()[0]

gdf_parcelles = gpd.read_postgis(
    f"SELECT id_parcel, {GEOM_COL} FROM derived.rpg_parcelles_aoi",
    conn,
    geom_col=GEOM_COL,
)
print(f"Parcelles chargées : {len(gdf_parcelles)}")

# Fusionner les géométries multi-lignes par id_parcel
gdf_parcelles = gdf_parcelles.dissolve(by="id_parcel", as_index=False)
print(f"Parcelles après dissolve : {len(gdf_parcelles)}")

# Label entier pour chaque parcelle (1-based, 0 = hors parcelle)
gdf_parcelles = gdf_parcelles.reset_index(drop=True)
gdf_parcelles["label"] = gdf_parcelles.index + 1
label_to_id = dict(zip(gdf_parcelles["label"], gdf_parcelles["id_parcel"]))

# Rasteriser les labels sur la grille AOI
shapes = list(zip(gdf_parcelles.geometry, gdf_parcelles["label"]))
label_grid = rio_rasterize(
    shapes,
    out_shape=(AOI_HEIGHT, AOI_WIDTH),
    transform=AOI_TRANSFORM,
    fill=0,
    dtype="int32",
)

n_lab = np.count_nonzero(label_grid)
print(f"Raster de labels : {AOI_HEIGHT} × {AOI_WIDTH}, "
      f"{n_lab:,} pixels étiquetés ({n_lab / label_grid.size * 100:.1f} %)")

In [ ]:
# ── QC : parcelles non capturées par la rasterisation des labels ─────

# Labels attendus vs labels effectivement présents dans le raster
labels_attendus  = set(gdf_parcelles["label"])
labels_presents  = set(np.unique(label_grid)) - {0}
labels_absents   = labels_attendus - labels_presents

gdf_absentes = gdf_parcelles[gdf_parcelles["label"].isin(labels_absents)].copy()

n_absentes   = len(gdf_absentes)
pct_absentes = n_absentes / len(gdf_parcelles) * 100

surface_absentes_ha = gdf_absentes.geometry.area.sum() / 1e4
surface_totale_ha   = gdf_parcelles.geometry.area.sum() / 1e4
pct_surface         = surface_absentes_ha / surface_totale_ha * 100

print(f"Parcelles non rasterisées : {n_absentes} sur {len(gdf_parcelles)} "
      f"({pct_absentes:.2f} %)")
print(f"Surface concernée : {surface_absentes_ha:.1f} ha sur "
      f"{surface_totale_ha:.0f} ha ({pct_surface:.3f} %)")

# Compacité (indice d'élongation) : périmètre² / (4π·surface)
# = 1 pour un cercle parfait, croît fortement pour les formes allongées
gdf_absentes["surface_ha"]  = gdf_absentes.geometry.area / 1e4
gdf_absentes["perimetre_m"] = gdf_absentes.geometry.length
gdf_absentes["compacite"]   = (
    gdf_absentes["perimetre_m"] ** 2
    / (4 * np.pi * gdf_absentes.geometry.area)
)

# Parcelles > 0,5 ha parmi les non rasterisées
grandes_absentes = (
    gdf_absentes[gdf_absentes["surface_ha"] > 0.5]
    .sort_values("compacite", ascending=False)
)
print(f"\nParcelles > 0,5 ha parmi les non rasterisées : {len(grandes_absentes)}")
print(grandes_absentes[["id_parcel", "surface_ha", "compacite"]].to_string(index=False))

> **Parcelles non rasterisées** : 2 751 parcelles sur 80 683 (3,41 %)
> ne capturent aucun centre de pixel à 20 m dans le raster de labels.
> Elles représentent 76,3 ha sur 334 910 ha (0,023 % de la surface totale).
> Parmi elles, 3 parcelles > 0,5 ha sont des bandes très allongées
> (compacité 11,0 à 94,4) — chemins d'exploitation ou bandes enherbées.
>
> **Parcelles orphelines** : 2 278 parcelles supplémentaires *sont*
> correctement rasterisées (elles capturent un centre de pixel), mais
> n'ont jamais reçu la moindre valeur S2 valide sur l'ensemble de la
> fenêtre d'observation — testé sur 106 à 152 dates selon la tuile,
> 100 % de pixels NaN à chaque fois. Cause identifiée : un angle mort
> structurel de la fauchée satellite, indépendant de la météo et de la
> date, localisé à la jonction des zones UTM 30N/31N que l'AOI
> chevauche. Ni un bug ni un déficit de couverture temporaire — une
> limite géométrique permanente des données sources.
>
> **Total exclu du feature set S3** : 5 029 parcelles sur 80 683
> (6,2 %), absentes de `derived.s2_parcelles_monthly`.

In [ ]:
# ── Statistiques zonales par parcelle (approche vectorisée) ──────────

def zonal_stats_from_labels(
    data: np.ndarray,
    labels: np.ndarray,
    label_to_id: dict,
) -> list[tuple]:
    """
    Calcule mean, std, p10, p90 par parcelle étiquetée.
    Retourne une liste de tuples (id_parcel, mean, std, p10, p90).

    Approche vectorisée : tri des pixels par label puis découpe —
    O(n log n) sur les pixels valides, bien plus rapide que
    80 689 appels rasterio.mask.
    """
    flat_data   = data.ravel()
    flat_labels = labels.ravel()

    valid = np.isfinite(flat_data) & (flat_labels > 0)
    d = flat_data[valid]
    l = flat_labels[valid]

    if len(d) == 0:
        return []

    order = np.argsort(l, kind="mergesort")
    d = d[order]
    l = l[order]

    unique_labels, start_idx = np.unique(l, return_index=True)
    groups = np.split(d, start_idx[1:])

    rows = []
    for lbl, pixels in zip(unique_labels, groups):
        n = len(pixels)
        if n == 0:
            continue
        rows.append((
            label_to_id[int(lbl)],
            float(np.mean(pixels)),
            float(np.std(pixels))  if n > 1 else 0.0,
            float(np.percentile(pixels, 10)),
            float(np.percentile(pixels, 90)),
        ))
    return rows

In [ ]:
# ── Boucle sur les composites : stats zonales + insertion PostGIS ────
mois_disponibles = sorted(
    d.name for d in COMPOSITES_DIR.iterdir()
    if d.is_dir() and len(d.name) == 7      # YYYY-MM
)
print(f"Mois disponibles : {len(mois_disponibles)}  {mois_disponibles}")

INSERT_SQL = """
    INSERT INTO derived.s2_parcelles_monthly
        (id_parcel, mois, variable, mean, std, p10, p90)
    VALUES %s
    ON CONFLICT (id_parcel, mois, variable) DO NOTHING
"""

total_ins = 0

for mois in mois_disponibles:
    n_mois = 0
    for variable in VARIABLES:
        tif_path = COMPOSITES_DIR / mois / f"{variable}.tif"
        if not tif_path.exists():
            print(f"  ⚠ {mois}/{variable} — fichier absent")
            continue

        with rasterio.open(tif_path) as src:
            data = src.read(1).astype(np.float32)
        data[data == -9999.0] = np.nan

        rows = zonal_stats_from_labels(data, label_grid, label_to_id)
        if not rows:
            continue

        insert_rows = [
            (r[0], mois, variable, r[1], r[2], r[3], r[4])
            for r in rows
        ]

        with conn.cursor() as cur:
            execute_values(cur, INSERT_SQL, insert_rows, page_size=5000)

        n_mois   += len(insert_rows)
        total_ins += len(insert_rows)

        del data, rows, insert_rows
        gc.collect()

    print(f"  {mois} — {n_mois:>8,} lignes  (cumul {total_ins:,})")

print(f"\nTerminé. {total_ins:,} lignes insérées.")

### 3.4 bis — Contrôle qualité

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
                SELECT mois, COUNT(DISTINCT variable) AS n_var, COUNT(DISTINCT id_parcel) AS n_parcelles
                FROM derived.s2_parcelles_monthly
                GROUP BY mois ORDER BY mois;
                """)
    for row in cur.fetchall():
        print(row)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(DISTINCT id_parcel)
        FROM derived.rpg_parcelles_aoi
        WHERE id_parcel NOT IN (SELECT DISTINCT id_parcel FROM derived.s2_parcelles_monthly);
    """)
    print(cur.fetchone()[0])

### 3.5 — Statistiques de complétude

Complète les statistiques spectrales de 3.4 avec un indicateur de
qualité par parcelle × mois : le nombre moyen de dates valides et le
pourcentage de la surface parcellaire effectivement couverte, dérivés
des rasters de complétude produits en 3.2 quater
(`data/completude/<YYYY-MM>_n_valid.tif`).

Utile pour nb04 : décider par mois × parcelle s'il faut garder,
imputer, ou exclure une feature — plus fin que le seuil global à
20 % de pixels AOI établi en 3.2 quater.

**Table** : `derived.s2_parcelles_completude` (clé `id_parcel × mois`),
indépendante de la variable spectrale — le masque de validité
(SCL + hors-fauchée) est partagé par toutes les variables issues
d'une même scène.

Comme 3.2 quater peut encore être en cours d'exécution dans un autre
kernel, cette section ne traite que les mois dont le raster de
complétude est déjà présent sur disque — les mois manquants seront
traités lors d'une réexécution ultérieure.

In [ ]:
# ── Table de complétude + fonction de stats zonales ───────────────────
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS derived.s2_parcelles_completude (
            id_parcel             TEXT  NOT NULL,
            mois                  TEXT  NOT NULL,
            n_dates_valides_moy   REAL,
            pct_pixels_couverts   REAL,
            PRIMARY KEY (id_parcel, mois)
        );
    """)
conn.commit()
print("Table derived.s2_parcelles_completude prête.")


def zonal_completude_from_labels(
    data: np.ndarray,
    labels: np.ndarray,
    label_to_id: dict,
) -> list[tuple]:
    """
    Calcule, par parcelle, le nombre moyen de dates valides et le
    pourcentage de pixels couverts (≥1 date valide) à partir du
    raster n_valid_mois (int16, 0 = pixel jamais valide ce mois-ci).
    Retourne une liste de tuples (id_parcel, n_dates_valides_moy, pct_pixels_couverts).
    """
    flat_data   = data.ravel().astype(np.float32)
    flat_labels = labels.ravel()

    valid = flat_labels > 0   # tous les pixels de parcelle, y compris n_valid == 0
    d = flat_data[valid]
    l = flat_labels[valid]

    if len(d) == 0:
        return []

    order = np.argsort(l, kind="mergesort")
    d = d[order]
    l = l[order]

    unique_labels, start_idx = np.unique(l, return_index=True)
    groups = np.split(d, start_idx[1:])

    rows = []
    for lbl, pixels in zip(unique_labels, groups):
        n = len(pixels)
        if n == 0:
            continue
        rows.append((
            label_to_id[int(lbl)],
            float(np.mean(pixels)),
            float(np.mean(pixels > 0) * 100),
        ))
    return rows

In [ ]:
# ── Boucle sur les rasters de complétude disponibles → stats zonales ──
mois_completude_dispo = sorted(
    p.name.replace("_n_valid.tif", "")
    for p in COMPLETUDE_DIR.glob("*_n_valid.tif")
)
mois_manquants = sorted(set(mois_complets) - set(mois_completude_dispo))

print(f"Mois de complétude disponibles : {len(mois_completude_dispo)}  {mois_completude_dispo}")
if mois_manquants:
    print(f"Mois manquants (3.2 quater pas encore arrivé jusque-là) : {mois_manquants}")

# ── Mois déjà en base — évite de recalculer ───────────────────────────
with conn.cursor() as cur:
    cur.execute("SELECT DISTINCT mois FROM derived.s2_parcelles_completude;")
    deja_en_base = {row[0] for row in cur.fetchall()}
print(f"{len(deja_en_base)} mois déjà en base — seront skippés")

INSERT_COMPLETUDE_SQL = """
    INSERT INTO derived.s2_parcelles_completude
        (id_parcel, mois, n_dates_valides_moy, pct_pixels_couverts)
    VALUES %s
    ON CONFLICT (id_parcel, mois) DO NOTHING
"""

total_ins = 0
n_skip = 0

for mois in mois_completude_dispo:
    if mois in deja_en_base:
        n_skip += 1
        continue

    tif_path = COMPLETUDE_DIR / f"{mois}_n_valid.tif"
    with rasterio.open(tif_path) as src:
        n_valid_mois = src.read(1)

    rows = zonal_completude_from_labels(n_valid_mois, label_grid, label_to_id)
    del n_valid_mois
    if not rows:
        print(f"  {mois} — aucune parcelle")
        continue

    insert_rows = [(r[0], mois, r[1], r[2]) for r in rows]

    with conn.cursor() as cur:
        execute_values(cur, INSERT_COMPLETUDE_SQL, insert_rows, page_size=5000)
    conn.commit()

    total_ins += len(insert_rows)
    print(f"  {mois} — {len(insert_rows):>8,} lignes  (cumul {total_ins:,})")

    del rows, insert_rows
    gc.collect()

print(f"\nTerminé. {total_ins:,} lignes insérées. {n_skip} mois skippés (déjà en base).")
if mois_manquants:
    print(f"Rappel : {len(mois_manquants)} mois restent à traiter une fois "
          f"3.2 quater terminé — relancer cette cellule à ce moment-là.")

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(DISTINCT id_parcel)
        FROM derived.s2_parcelles_completude
        WHERE id_parcel = ANY(%s) AND n_dates_valides_moy > 0;
    """, (orphelines,))
    n_incoherent = cur.fetchone()[0]
print(f"Parcelles orphelines avec n_dates_valides_moy > 0 (incohérence attendue à 0) : {n_incoherent}")

### 3.6 — Agrégation zonale NDVI aux dates d'acquisition

En complément des composites mensuels (section 3.4), cette section
agrège le NDVI par parcelle **à chaque date d'acquisition**, sans
compositage temporel. L'objectif est de disposer d'un profil NDVI
irrégulièrement échantillonné mais fidèle à la dynamique réelle du
couvert — nécessaire à l'extraction des métriques phénologiques
(SOS/POS/EOS, sprint S4) que le pas mensuel écraserait.

**Source** : les GeoTIFF `{scene_id}_NDVI.tif` produits scène par scène
en 3.2 (déjà masqués SCL), conservés sur disque. Chaque scène est
reprojetée sur la grille AOI 20 m puis agrégée par le raster de labels
construit en 3.4.

**Table** : `derived.s2_parcelles_ndvi_dates` (clé `id_parcel × date`),
avec `mean`, `std` et `n_pixels` — ce dernier permettant de filtrer les
parcelles à faible couverture valide lors du lissage phénologique.

In [ ]:
# ── Table NDVI par date + inventaire des scènes NDVI ──────────────────
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS derived.s2_parcelles_ndvi_dates (
            id_parcel  TEXT  NOT NULL,
            date       DATE  NOT NULL,
            mean       REAL,
            std        REAL,
            n_pixels   INTEGER,
            PRIMARY KEY (id_parcel, date)
        );
    """)
conn.commit()
print("Table derived.s2_parcelles_ndvi_dates prête.")

# Inventaire des NDVI par scène sur toutes les tuiles
ndvi_files = sorted(INDICES_DIR.rglob("*_NDVI.tif"))
print(f"Fichiers NDVI par scène : {len(ndvi_files)}")

# Extraction de la date depuis le scene_id (champ 3 : YYYYMMDDTHHMMSS)
def date_from_ndvi_path(p: Path) -> str:
    scene_id = p.name.replace("_NDVI.tif", "")
    return scene_id.split("_")[2][:8]  # YYYYMMDD

# Regroupement par date : une date peut avoir plusieurs tuiles/scènes
from collections import defaultdict
scenes_par_date = defaultdict(list)
for p in ndvi_files:
    scenes_par_date[date_from_ndvi_path(p)].append(p)

print(f"Dates distinctes : {len(scenes_par_date)}")

In [ ]:
# ── Boucle sur les dates : mosaïque NDVI → stats zonales → PostGIS ────
def zonal_ndvi_from_labels(data, labels, label_to_id):
    """mean, std, n_pixels par parcelle. Retourne (id_parcel, mean, std, n)."""
    flat_d, flat_l = data.ravel(), labels.ravel()
    valid = np.isfinite(flat_d) & (flat_l > 0)
    d, l = flat_d[valid], flat_l[valid]
    if len(d) == 0:
        return []
    order = np.argsort(l, kind="mergesort")
    d, l = d[order], l[order]
    unique_labels, start_idx = np.unique(l, return_index=True)
    groups = np.split(d, start_idx[1:])
    rows = []
    for lbl, px in zip(unique_labels, groups):
        n = len(px)
        rows.append((
            label_to_id[int(lbl)],
            float(np.mean(px)),
            float(np.std(px)) if n > 1 else 0.0,
            int(n),
        ))
    return rows

INSERT_NDVI_SQL = """
    INSERT INTO derived.s2_parcelles_ndvi_dates
        (id_parcel, date, mean, std, n_pixels)
    VALUES %s
    ON CONFLICT (id_parcel, date) DO NOTHING
"""

total_ins = 0
for i, (date_str, paths) in enumerate(sorted(scenes_par_date.items()), 1):
    date_iso = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"

    # Mosaïque inter-tuiles : médiane pixel à pixel des scènes du jour sur AOI
    stack = np.full((len(paths), AOI_HEIGHT, AOI_WIDTH), np.nan, dtype=np.float32)
    for k, p in enumerate(paths):
        stack[k] = reproject_to_aoi(p)
    with np.errstate(all="ignore"):
        daily = np.nanmedian(stack, axis=0)
    del stack

    rows = zonal_ndvi_from_labels(daily, label_grid, label_to_id)
    del daily
    if not rows:
        print(f"  [{i}/{len(scenes_par_date)}] {date_iso} — aucune parcelle")
        continue

    insert_rows = [(r[0], date_iso, r[1], r[2], r[3]) for r in rows]
    with conn.cursor() as cur:
        execute_values(cur, INSERT_NDVI_SQL, insert_rows, page_size=5000)
    conn.commit()

    total_ins += len(insert_rows)
    print(f"  [{i}/{len(scenes_par_date)}] {date_iso} "
          f"({len(paths)} scène·s) — {len(insert_rows):>7,} lignes  "
          f"(cumul {total_ins:,})")
    del rows, insert_rows
    gc.collect()

print(f"\nTerminé. {total_ins:,} lignes insérées dans s2_parcelles_ndvi_dates.")

### 3.6 bis - Controle qualite

In [ ]:
# ── QC 3.6 : cohérence de s2_parcelles_ndvi_dates ─────────────────────
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*) AS n_lignes,
               COUNT(DISTINCT id_parcel) AS n_parcelles,
               COUNT(DISTINCT date) AS n_dates,
               MIN(n_pixels) AS min_pixels,
               MAX(n_pixels) AS max_pixels,
               AVG(n_pixels) AS moy_pixels
        FROM derived.s2_parcelles_ndvi_dates;
    """)
    n_lignes, n_parcelles, n_dates, min_px, max_px, moy_px = cur.fetchone()

print(f"Lignes           : {n_lignes:,}")
print(f"Parcelles         : {n_parcelles:,} (max attendu ~77 932)")
print(f"Dates distinctes  : {n_dates} (attendu 166)")
print(f"n_pixels min/max/moy : {min_px} / {max_px} / {moy_px:.1f}")

# Doublons sur la clé primaire (id_parcel, date) ?
with conn.cursor() as cur:
    cur.execute("""
        SELECT id_parcel, date, COUNT(*)
        FROM derived.s2_parcelles_ndvi_dates
        GROUP BY id_parcel, date
        HAVING COUNT(*) > 1
        LIMIT 5;
    """)
    doublons = cur.fetchall()
print(f"Doublons détectés : {len(doublons)} (attendu 0)")

OPERATION SPECIALE